In [11]:
import pandas as pd
import numpy as np
import os
import re

class ProductRanker:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None

    # -----------------------------
    # Load File
    # -----------------------------
    def load_data(self):
        if not os.path.exists(self.file_path):
            raise FileNotFoundError("File not found")

        if self.file_path.endswith('.xlsx'):
            self.df = pd.read_excel(self.file_path)
        elif self.file_path.endswith('.csv'):
            self.df = pd.read_csv(self.file_path)
        else:
            raise ValueError("Unsupported file format")

    # -----------------------------
    # Column Mapping (Flexible)
    # -----------------------------
    def map_columns(self):
        cols = {c.lower(): c for c in self.df.columns}

        def find_col(possible):
            for p in possible:
                for c in cols:
                    if p in c:
                        return cols[c]
            return None

        self.col_rating = find_col(["rating"])
        self.col_reviews = find_col(["review", "count"])
        self.col_price = find_col(["price", "cost"])
        self.col_name = find_col(["product", "name", "title"])

        if not all([self.col_rating, self.col_reviews, self.col_price, self.col_name]):
            raise ValueError("Columns not detected automatically")

    # -----------------------------
    # Cleaning Logic
    # -----------------------------
    def extract_number(self, val):
        val = str(val).upper()

        # extract number with optional decimal
        match = re.search(r"\d+(\.\d+)?", val)
        if not match:
            return np.nan

        num = float(match.group())

        if "K" in val:
            num *= 1000
        elif "M" in val:
            num *= 1_000_000

        return num

    def clean_rating(self):
        self.df["Rating"] = self.df[self.col_rating].apply(self.extract_number)

    def clean_reviews(self):
        self.df["Reviews"] = self.df[self.col_reviews].apply(self.extract_number)

    def clean_price(self):
        def parse_price(val):
            val = str(val)
    
            # remove everything except digits and dots
            val = re.sub(r"[^\d.]", "", val)
    
            # handle empty
            if val == "":
                return np.nan
    
            try:
                return float(val)
            except:
                return np.nan
    
        self.df["Price"] = self.df[self.col_price].apply(parse_price)

    # -----------------------------
    # Handle Missing Values
    # -----------------------------
    def handle_missing(self):
        self.df["Rating"].fillna(self.df["Rating"].mean(), inplace=True)
        self.df["Reviews"].fillna(0, inplace=True)
        self.df["Price"].fillna(self.df["Price"].median(), inplace=True)

    # -----------------------------
    # Weighted Rating
    # -----------------------------
    def compute_score(self):
        C = self.df["Rating"].mean()
        m = max(self.df["Reviews"].quantile(0.75), 1)

        self.df["WeightedRating"] = (
            (self.df["Reviews"] / (self.df["Reviews"] + m)) * self.df["Rating"] +
            (m / (self.df["Reviews"] + m)) * C
        )

    # -----------------------------
    # Output
    # -----------------------------
    def get_top_products(self, n=10):
        result = self.df.sort_values(by="WeightedRating", ascending=False)
        return result[[self.col_name, "WeightedRating", "Rating", "Reviews", "Price"]].head(n)

    # -----------------------------
    # Pipeline
    # -----------------------------
    def run(self):
        self.load_data()
        self.map_columns()
        self.clean_rating()
        self.clean_reviews()
        self.clean_price()
        self.handle_missing()
        self.compute_score()


# -----------------------------
# Usage
# -----------------------------
if __name__ == "__main__":
    file_path = "data4.xlsx"

    ranker = ProductRanker(file_path)
    ranker.run()

    top = ranker.get_top_products(10)
    print(top)

    top.to_excel("ranked_products.xlsx", index=False)


                                         Product Name  WeightedRating  Rating  \
66  Corsair Vengeance LPX 16GB (2 X 8GB) DDR4 3600...        4.770527     4.8   
57  Patriot Memory 16GB(2x8GB) Viper III DDR3 1866...        4.622797     4.7   
65  Patriot Memory 16GB(2x8GB) Viper III DDR3 1600...        4.605641     4.7   
22  Corsair Vengeance LPX 8GB (1x8GB) DDR4 3200MHZ...        4.522711     4.6   
26  Crucial RAM 16GB DDR4 3200 MHz CL22 Laptop Mem...        4.497608     4.5   
48  Kingston Fury Impact 32GB 3200MHz DDR4 CL20 La...        4.482984     4.5   
13  G.SKILL Trident Z RGB 16GB (2 * 8GB) DDR4 3200...        4.476356     4.5   
46  CORSAIR Vengeance RGB DDR5 RAM 32GB (2x16GB) 6...        4.458863     4.6   
25  XPG ADATA GAMMIX D30 DDR4 8GB (1x8GB) 3200MHz ...        4.452209     4.5   
53  Acer SD100 8GB Single RAM 3200 MHz DDR4 CL22 1...        4.437692     4.6   

    Reviews    Price  
66  16800.0  13689.0  
57   4500.0   5049.0  
65   3500.0   5049.0  
22   3200.0   76

C:\Users\SUMIT\AppData\Local\Temp\ipykernel_21456\3281901125.py:94: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.df["Rating"].fillna(self.df["Rating"].mean(), inplace=True)
C:\Users\SUMIT\AppData\Local\Temp\ipykernel_21456\3281901125.py:95: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves a